# Parameters 

In [ ]:
from datetime import datetime, date
import datetime
import pandas as pd
import os
from sklearn.metrics import mean_squared_error, mean_absolute_error
from pmdarima import auto_arima
pd.set_option("display.max_rows", None)
import numpy as np
import fnmatch
import logging
pd.set_option('display.float_format', '{:,.2f}'.format)
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)

# The data source is coming from https://www.kaggle.com/datasets/yasserh/walmart-dataset

# Handle to write logs to a file
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.DEBUG)
logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s - %(levelname)s - %(message)s'
)



exog_vars = ['Holiday_Flag', 'Temperature', 'Fuel_Price',
       'CPI', 'Unemployment'] 


# parameters
startYearPrediction   = 2012
# We need to specify the start week of prediction1
StartWeekPredict = 4


# --------------------------------OBSERVATION------------------------------------------------------------------------------------
# we don´t recommend changing this parameters because the backtesting has been set to be 4 weeks, we have added them here just as a reminder that we can adjust that 
# Number of weeks to predict,let´s say that you want to predict from the first week to the third week  to need to set 3
NumberWeekPredict = 4
# Number of weeeks that we want to set for the backtesting in production
BackTestingWeek = 4
# ------------------------------END OBSERVATION---------------------------------------------------------------------------------------------


# This process may take around 24 hours because we go throught different combination of the features and look for the best hyperparameter in each iteration
# the output of this flag is Feature_Selection.xlsx
flag_cal_best_features = True

# In case if we have changed the cluster assignation or the the best feature combination we can recalculate the hyperparameter for each cluster using this flag
# the output for this flag is the file Cluster_Hyperparam_sarima.xlsx , it doesn´t save historical hyperparameter, it is a possible improvement. 
flag_hyper_param_sarima  = False

# If we want to reassing the clusters we can this flag, it does not have historical data so it´ll overwrite the Cluster_Evaluation file 
# there is a possible improvement, currently we are using all the feature to determine the cluster creation.
flag_cluster_assigment = False


# Parameter transformation
startDatePrediction = datetime.date.fromisocalendar(startYearPrediction, StartWeekPredict, 5)
endDatePrediction= startDatePrediction + pd.DateOffset(weeks=NumberWeekPredict)
startDatePrediction = pd.to_datetime(startDatePrediction)

endDatePrediction = pd.to_datetime(endDatePrediction)

output_folder_cluster = 'Cluster_Evaluation'
os.makedirs(output_folder_cluster, exist_ok=True)

# readCsv

In [ ]:
dfWalmartSales_original = pd.read_csv(os.path.join(os.getcwd(), 'Walmart.csv'))  

dfWalmartSales_original['Date'] = pd.to_datetime(dfWalmartSales_original['Date'], format='%d-%m-%Y')

dfWalmartSalesClusters = dfWalmartSales_original.copy()

dfWalmartSales = dfWalmartSales_original.copy()

dfWalmartSalesFinalCluster = dfWalmartSales_original.copy()

# ClusterCreation

## ParameterCreation
we group the informacion per store and week to use it to predict the cluster
dfWalmartSalesClusters it's a copy of the original dataset to avoid affecting the dataset that we are going to use in the train split step 

In [ ]:

def cluster_parameters(dfWalmartSales,dfWalmartSalesClusters,exog_vars):

    dfExog = {'Temperature': 'mean', 'Fuel_Price': 'mean', 'CPI': 'mean', 'Unemployment': 'mean'}

    dict_fil = {k: v for k, v in dfExog.items() if k in exog_vars}

    dfWalmartSalesClusters['month'] = dfWalmartSalesClusters['Date'].dt.month
    dfWalmartSalesClusters['week_number'] = dfWalmartSalesClusters['Date'].dt.isocalendar().week

    dfWalmartSalesExogenas = dfWalmartSales.groupby(['Store']).agg(dict_fil).reset_index()

    dfWalmartSalesClusters = dfWalmartSalesClusters.groupby(['Store', 'week_number']).agg({'Weekly_Sales': 'mean'}).reset_index()

    dfWalmartSalesClusters = dfWalmartSalesClusters.pivot(
        index="Store",
        columns="week_number",
        values=["Weekly_Sales"] 
    )
    dfWalmartSalesClusters.index.name = None
    dfWalmartSalesClusters.columns.name = None
    
    dfWalmartSalesClusters.columns = list(range(1, 53))
    dfWalmartSalesClusters.index.name = 'Store'
    dfWalmartSalesClusters = dfWalmartSalesClusters.reset_index()
    dfWalmartSalesClusters = dfWalmartSalesClusters.merge(dfWalmartSalesExogenas, how='left', on='Store')
    return dfWalmartSalesClusters


## TrainingPredictCluster

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN

def clusterPredict(dfWalmartSales,dfWalmartSalesClusters):
    
    scaler = StandardScaler()

    dbscan = DBSCAN(eps=2, min_samples=2) 

    clusters = dbscan.fit_predict(scaler.fit_transform(dfWalmartSalesClusters.drop(columns=['Store']).to_numpy()))
    pd.DataFrame(clusters, columns=['Cluster']).value_counts().sort_index()

    dfClusterLabel = pd.DataFrame(clusters, columns=['cluster_label'])
    dfClusterLabel  = pd.concat([dfWalmartSalesClusters, dfClusterLabel], axis= 1)
    dfClusterLabel = dfClusterLabel[['Store', 'cluster_label']]
    dfWalmartSales = dfWalmartSales.merge(dfClusterLabel, how='left', on='Store')
    return dfWalmartSales



def clusterPredictEpsilon(dfWalmartSales,dfWalmartSalesClusters,eps,min_samples):
    
    scaler = StandardScaler()

    dbscan = DBSCAN(eps=eps, min_samples= min_samples) 

    clusters = dbscan.fit_predict(scaler.fit_transform(dfWalmartSalesClusters.drop(columns=['Store']).to_numpy()))
    pd.DataFrame(clusters, columns=['Cluster']).value_counts().sort_index()

    dfClusterLabel = pd.DataFrame(clusters, columns=['cluster_label'])
    dfClusterLabel  = pd.concat([dfWalmartSalesClusters, dfClusterLabel], axis= 1)
    dfClusterLabel = dfClusterLabel[['Store', 'cluster_label']]
    dfWalmartSales = dfWalmartSales.merge(dfClusterLabel, how='left', on='Store')
    return dfWalmartSales




## ClusterGraphPDF

In [ ]:

def ClusterGraphPDF(dfWalmartSales):
    import matplotlib.pyplot as plt
    import os
    # Folder where the images will be saved
    output_folder_cluster = 'Cluster_Evaluation'
    os.makedirs(output_folder_cluster, exist_ok=True)

    clusters_unicos = dfWalmartSales['cluster_label'].unique()

    # deleting previous cluster evaluation information
    for i in os.listdir(output_folder_cluster):
         os.remove(os.path.join(output_folder_cluster,i))

    for cluster_id in clusters_unicos:
        plt.figure(figsize=(8, 4))
        
        cluster_data = dfWalmartSales[dfWalmartSales['cluster_label'] == cluster_id]
        stores_in_cluster = cluster_data['Store'].unique()
        
        for store in stores_in_cluster:
            store_data = cluster_data[cluster_data['Store'] == store]
            plt.plot(store_data['Date'], store_data['Weekly_Sales'], 
                    alpha=0.5, linewidth=0.8)
        
        # Cluster mean
        avg_sales = cluster_data.groupby('Date')['Weekly_Sales'].mean()
        plt.plot(avg_sales.index, avg_sales.values, 
                color='red', linewidth=3, label='Cluster Mean')
        
        plt.title(f'Cluster {cluster_id+1} - {len(stores_in_cluster)} Stores')
        plt.xlabel('Fecha')
        plt.ylabel('Ventas Semanales')
        plt.legend()
        plt.xticks(rotation=45)
        plt.tight_layout()
        
        # --- Save the image ---
        plt.savefig(f'{output_folder_cluster}/cluster_{cluster_id+1}.png', dpi=150, bbox_inches='tight')
        plt.close()  # Close the figure to release memory

    print(f"✅ {len(clusters_unicos)} images were saved in '{output_folder_cluster}'")


    from PIL import Image
    import os

    # Folder where the imagenes are
    output_folder_cluster = 'Cluster_Evaluation'

    # Obtener todas las imágenes de los clusters
    imagenes = [
        f for f in os.listdir(output_folder_cluster)
        if f.endswith('.png') and f.startswith('cluster_')
    ]

    # we order the cluster by their number 
    imagenes = sorted(
        imagenes,
        key=lambda x: int(x.replace('cluster_', '').replace('.png', ''))
    )

    # we open the imagene 
    imagenes_pil = [
        Image.open(os.path.join(output_folder_cluster, img)).convert('RGB')
        for img in imagenes
    ]

    # Name of the cluster
    pdf_path = os.path.join(output_folder_cluster, 'All_Clusters.pdf')

    # PDF creation 
    imagenes_pil[0].save(
        pdf_path,
        save_all=True,
        append_images=imagenes_pil[1:]
    )

    print(f"✅ PDF created: {pdf_path}")

## iterationChooseBestHyperParameterCluster

In [ ]:

import json

from sklearn.metrics import silhouette_score, calinski_harabasz_score

if flag_cluster_assigment == True:
    bitIterationEpsilon = []

    startEpsilon = 1.1
    for m in range(1,2):

        for i in range(1,15):
            startEpsilon = round(startEpsilon+0.1,2)
            
            dfWalmartPerStore = cluster_parameters(dfWalmartSales,dfWalmartSalesClusters,exog_vars)
            dfWalmartSalesClusterlabel = clusterPredictEpsilon(dfWalmartSales,dfWalmartPerStore,startEpsilon,m)
            NumStore = dfWalmartPerStore['Store'].nunique()
            Start_Date = dfWalmartSales['Date'].min().strftime('%Y-%m-%d')
            End_Date = dfWalmartSales['Date'].max().strftime('%Y-%m-%d')
            
            dfWalmartPerStore = dfWalmartPerStore.merge(dfWalmartSalesClusterlabel[['Store', 'cluster_label' ]].drop_duplicates(), how='left', on='Store')
            
            scaler = StandardScaler()

            score = silhouette_score(scaler.fit_transform(dfWalmartPerStore.drop(columns=['Store','cluster_label']).to_numpy()), dfWalmartPerStore['cluster_label'])
            
            scoreCalinski = calinski_harabasz_score(dfWalmartPerStore.drop(columns=['Store','cluster_label']).to_numpy() ,  dfWalmartPerStore['cluster_label'] )
            # print()

            # Code to assign a number to the store that does not belong to any cluster -1
            maxNumCluster = dfWalmartPerStore['cluster_label'].max()
            
            dfWithoutCluster = dfWalmartPerStore[dfWalmartPerStore['cluster_label']==-1]
            if len(dfWithoutCluster)>0:
                dfWithCluster = dfWalmartPerStore[dfWalmartPerStore['cluster_label']!=-1]
                dfWithoutCluster = dfWithoutCluster.reset_index(drop=True)
                dfWithoutCluster['cluster_label'] = dfWithoutCluster.index.values + 1 + maxNumCluster
                dfClusterUnion = pd.concat([dfWithoutCluster,dfWithCluster])
                
            else:
                dfClusterUnion = dfWalmartPerStore.copy()
        
            NClusters = dfClusterUnion['cluster_label'].nunique()
            # print(f'Epsilon value:{startEpsilon}',"Silhouette Score:", round(score,6), 'Clusters:',NClusters)

            bitIterationEpsilon.append({'Start_Date':Start_Date,'End_Date':End_Date, 'features':json.dumps(exog_vars), 'Epsilon_value':startEpsilon,"min_samples":m, 
                                        'Stores':NumStore,'Clusters':NClusters, "Silhouette Score":round(score,6),  "Calinski Score":round(scoreCalinski,6) })
            
    
    bitIterationEpsilon = pd.DataFrame(bitIterationEpsilon)
    bitIterationEpsilon = bitIterationEpsilon.sort_values(by='Silhouette Score', ascending=False).reset_index(drop=True)
    bitIterationEpsilon['Choosen'] = np.where(bitIterationEpsilon.index.values == 0, True, False)
    bitIterationEpsilon['Date_run'] = pd.to_datetime('now').strftime('%Y-%m-%d')

    Index_values = ['Date_run','Start_Date', 'End_Date','Stores', 'features', 'Epsilon_value', 'min_samples', 'Clusters', 'Silhouette Score',
        'Calinski Score', 'Choosen']

    bitIterationEpsilon  = bitIterationEpsilon[Index_values]

    ClusterGraphPDF(dfWalmartSalesClusterlabel)


    ## ExportClusterLabel

    # we are going to retrieve the best parameter for the cluster evaluation
    bestParameterCluster = bitIterationEpsilon[bitIterationEpsilon['Choosen'] == True ]
    MinSampleFinal = bestParameterCluster['min_samples'].values[0]
    FeaturesFinal = json.loads(bestParameterCluster['features'].values[0])
    epsFinal = bestParameterCluster['Epsilon_value'].values[0]
    dfSalesPerStoreFinal = cluster_parameters(dfWalmartSales,dfWalmartSalesClusters,FeaturesFinal)
    dfsalesClusterlabelFinal = clusterPredictEpsilon(dfWalmartSalesFinalCluster,dfSalesPerStoreFinal,epsFinal,MinSampleFinal)
    dfSalesPerStoreFinal = dfSalesPerStoreFinal.merge(dfsalesClusterlabelFinal[['Store', 'cluster_label' ]].drop_duplicates(), how='left', on='Store')

    # We shouldnt have any store without cluster, because we have chosen at least one store per cluster, but we are going to check it anyway.
    dfWithoutCluster = dfSalesPerStoreFinal[dfSalesPerStoreFinal['cluster_label']==-1]
    if len(dfWithoutCluster)>0:
        logger.info(f"Stores without cluster: {len(dfWithoutCluster)}")
        raise ValueError(f"Stores without cluster: {len(dfWithoutCluster)}. Please check the clustering parameters.")
        
    dfSalesPerStoreFinal['cluster_label']  = dfSalesPerStoreFinal['cluster_label'].astype(int) + 1 

    dfSalesPerStoreFinal =dfSalesPerStoreFinal[['Store','cluster_label','Temperature',    'Fuel_Price',           'CPI',
            'Unemployment' ]]
    dfSalesPerStoreFinal.columns = ['Store','cluster_label','Mean_Temperature',    'Mean_Fuel_Price',           'Mean_CPI',  'Mean_Unemployment']
    dfSalesPerStoreFinal

    with pd.ExcelWriter(os.path.join(output_folder_cluster,"clust_hparam_eval.xlsx"), engine="openpyxl" ) as writer:

        bitIterationEpsilon.to_excel(
                writer,
                sheet_name="Cluster_Evaluation",
    

                index=True
            )
        dfSalesPerStoreFinal.to_excel(
                writer, sheet_name="Stores_Cluster_Assignment",
        
            
                
                            index=True)



# Principal Functions

## splitTrainingTest

In [ ]:
def trainTest(df,start_date_p, end_date_p ):
    dfWalmartSalesStore1= df.copy()
    dfTrainingTest  = dfWalmartSalesStore1.reset_index()
    dfTrainingTest['Date'] = pd.to_datetime(dfTrainingTest['Date'])
    StartTrainingData = start_date_p - pd.DateOffset(weeks=104)
    EndTrainingData = start_date_p - pd.DateOffset(weeks=1)

    logger.info(f"StartTrainingData: {StartTrainingData}")
    logger.info(f"EndTrainingData: {EndTrainingData}")
    train = dfTrainingTest[(dfTrainingTest['Date']>= StartTrainingData) & (dfTrainingTest['Date']<= EndTrainingData)]
    test = dfTrainingTest[(dfTrainingTest['Date']>=start_date_p) & (dfTrainingTest['Date']<= end_date_p)]
    train.sort_values(by='Date', ascending=True, inplace=True)
    test.sort_values(by='Date', ascending=True, inplace=True)
    return train, test

    
    

## adjustingData

In [ ]:
# After we have assigned the cluster to the store we need to group the features by cluster

def adjustingData(df):
    
    dfadjusted = df.groupby('Date').agg({'Weekly_Sales': 'sum','Temperature': 'mean', 'Fuel_Price': 'mean', 'CPI': 'mean', 'Unemployment': 'mean','Holiday_Flag':'sum'}).reset_index()
    dfadjusted.sort_values(by='Date', ascending=True, inplace=True)
    return dfadjusted


## BestHyperparameters
Based on the trend graph, we initially treated all observed trends as seasonality. However, in a real-world project it is essential to validate this assumption to ensure that the patterns truly represent seasonal effects rather than other underlying factors.

In [ ]:

def bestHyperparameter(train):

    bestParameterModel = auto_arima(
        train["Weekly_Sales"], 
        seasonal=True, 
        m=52,          # yearly seasonality in weekly data
        d=None,        # let auto_arima decide non-seasonal differencing
        D=1,           # seasonal differencing
        start_p=0, start_q=0, start_P=0, start_Q=0,  # safe starting values
        max_p=1, max_q=1, max_P=1, max_Q=1,          # keep search space small
        stepwise=True,                                # heuristic search
        suppress_warnings=True,
        error_action="ignore",
        trace=False                                   # turn off verbose printing
    )
    return bestParameterModel




## exogenousCalculation
We are asumming that we dont have access to exogeous values so we are calculating an average for Fuel_price, CPI and unemployment from the last 12 months.
For temperature and holiday we are the previous´s year information. 

In [ ]:

def exogenousCalculation(startYearPrediction, StartWeekPredict,NumberWeekPredict, train):
       ExogYearPrediction = startYearPrediction- 1 
       startExogDate = pd.to_datetime( datetime.date.fromisocalendar(ExogYearPrediction, StartWeekPredict, 5))
       EndExogDate= startExogDate + pd.DateOffset(weeks=NumberWeekPredict)



       print(startExogDate)
       print(EndExogDate)
       # here we are getting the data from the previous year       
       dfExog = train[(train['Date']>=startExogDate )& (train['Date']<=EndExogDate)][['Temperature','Holiday_Flag']]
       # here we are calculating the mean for the exogenous variables that we dont have access to, like Fuel_Price, CPI and Unemployment      
       dfMeanExog = train[train['Date']>=(startExogDate)][['Fuel_Price','CPI','Unemployment']].mean()

       dfExog['Fuel_Price'] = dfMeanExog['Fuel_Price']
       dfExog['CPI'] = dfMeanExog['CPI']
       dfExog['Unemployment'] = dfMeanExog['Unemployment']
       train = train[['Weekly_Sales', 'Holiday_Flag', 'Temperature', 'Fuel_Price',
              'CPI', 'Unemployment']]
       return dfExog , train 



## TrainingModelForecast
In this step the model is trained to predict new values either for training, backtesting or selecting the best features. 

In [ ]:
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX


# This function is for the best feature selection processs
def trainingForcast(bestParameterModel,dfExog,exog_vars,NumberWeekPredict,train):
    # NumberWeekPredict = NumberWeekPredict + 1
    historical_exog = train[exog_vars]    
    model = SARIMAX(
        train['Weekly_Sales'], 
        exog=historical_exog,           # Pass your historical external variables
        order= bestParameterModel.get_params()['order'],                # (p, d, q) - auto-regressive parameters
        seasonal_order=bestParameterModel.get_params()['seasonal_order'],   # (P, D, Q, S) - S=52 for yearly seasonality
        S= 52,
        enforce_stationarity=False,
        enforce_invertibility=False
    ).fit()
    # Ensuring that we are going to send the right order of the exogenous variables

    dfExog = dfExog[exog_vars]

    forecast = model.forecast(steps=NumberWeekPredict, exog=dfExog)
    return forecast


# This function is for the  production process

def forecastPrd(orderParameter, seasonalParameter,dfExog,exog_vars,NumberWeekPredict,train):
  
    historical_exog = train[exog_vars]    
    model = SARIMAX(
        train['Weekly_Sales'], 
        exog=historical_exog,           # Pass your historical external variables
        order= orderParameter,                # (p, d, q) - auto-regressive parameters
        seasonal_order=seasonalParameter,   # (P, D, Q, S) - S=52 for yearly seasonality
        S= 52,
        enforce_stationarity=False,
        enforce_invertibility=False
    ).fit()
    # Ensuring that we are going to send the right order of the exogenous variables

    dfExog = dfExog[exog_vars]

    forecast = model.forecast(steps=NumberWeekPredict, exog=dfExog)
    return forecast


## Metrics
This function was used to determine the best combination of feature and in production is used for the backtesting

In [ ]:
# from numpy import test
from sklearn.metrics import mean_absolute_percentage_error
def metricts(test,NumberWeekPredict, forecast):
        rmse = np.sqrt(mean_squared_error(test[[ 'Weekly_Sales' ]][:NumberWeekPredict], forecast))

        print(f"RMSE: {rmse:.2f}")

        mape = mean_absolute_percentage_error(test[[ 'Weekly_Sales' ]][:NumberWeekPredict], forecast)

        print(f"MAPE: {mape:.4f}")
        print(f"MAPE (%): {mape * 100:.2f}%")

        mae = mean_absolute_error(test[[ 'Weekly_Sales' ]][:NumberWeekPredict], forecast)
        predicted_value  =  forecast.reset_index(drop=True).sum()
        expected_value = test[[ 'Weekly_Sales' ]][:NumberWeekPredict].reset_index(drop=True)['Weekly_Sales'].sum() 
        print(f"MAE: {mae:.2f}")
        return rmse, mape, mae,predicted_value, expected_value


# BestFeatureSelection
Different feature combinations were evaluated using a 4‑week horizon, and the combination with the lowest MAE was selected

## Iteration

In [ ]:


if flag_cal_best_features == True:


    controls = []
    controls2 = []
    baseDfPrediction  = pd.DataFrame({}, columns=['date','cluster','forecast'])
    
    listDates = pd.date_range(startDatePrediction , periods=4,freq='W-FRI')

    from itertools import combinations


    exog_vars = ['Holiday_Flag', 'Temperature', 'Fuel_Price',
        'CPI', 'Unemployment'] 

    result = []
    for r in range(4, len(exog_vars) + 1):
        result.extend(combinations(exog_vars, r))
    result
    result = [list(x) for x in result]

    print(result)

    for exog_vars_ in result:
        dfWalmartSalesClusters = dfWalmartSales_original.copy()
        dfWalmartSales = dfWalmartSales_original.copy()
        
        dfWalmartSalesClusters = cluster_parameters(dfWalmartSales,dfWalmartSalesClusters,exog_vars_)
    
        dfWalmartSales = clusterPredict(dfWalmartSales,dfWalmartSalesClusters)

        for i in listDates:
            startDatePrediIteration = i 
            endDatePredicIteration = startDatePrediIteration + pd.DateOffset(weeks=NumberWeekPredict)
            for cluster in dfWalmartSales['cluster_label'].unique():
                dfWalmartSalesStore1  = dfWalmartSales[dfWalmartSales['cluster_label'] == cluster]
                dfWalmartSalesStore1 = adjustingData(dfWalmartSalesStore1)
                
                train, test = trainTest(dfWalmartSalesStore1,startDatePrediIteration, endDatePredicIteration)
                bestParameterModel = bestHyperparameter(train)   
                dfExog, train = exogenousCalculation(startDatePrediIteration.year, startDatePrediIteration.weekofyear,NumberWeekPredict, train)
                dfExog =dfExog[:NumberWeekPredict]
                forecast = trainingForcast(bestParameterModel,dfExog,exog_vars_,NumberWeekPredict,train)

                test.reset_index(drop=True, inplace=True)
                test2Months = test.loc[[0,1]]
                forecast.reset_index(drop=True, inplace=True)
                forecast2Months = forecast.loc[[0,1]]
                rmse, mape, mae,predicted_value, expected_value= metricts(test,NumberWeekPredict, forecast)
                rmse2, mape2, mae2,predicted_value2, expected_value2= metricts(test2Months,2, forecast2Months)
                # we are creating a df for each cluster to see the prediction    
                transposed_df = pd.DataFrame({'date': pd.date_range(start=startDatePrediIteration, periods=NumberWeekPredict, freq='W')})
                transposed_df = pd.concat([transposed_df,forecast.reset_index().drop(columns = {'index'})],axis=1, ignore_index=True)
                transposed_df.columns = ['date', 'forecast']
                transposed_df['cluster'] = cluster + 1
                baseDfPrediction = pd.concat([baseDfPrediction, transposed_df], ignore_index=True)
                transposed_df = transposed_df[['date','cluster','forecast']]
                control = {'date_run': pd.to_datetime('now').strftime('%Y-%m-%d'), 'cluster': cluster, 'startDatePrediction': startDatePrediIteration, 'NumberWeekPredict': NumberWeekPredict,'exog_vars':exog_vars_ ,'rmse': rmse, 'mape': mape, 'mae': mae}
                control2 = {'date_run': pd.to_datetime('now').strftime('%Y-%m-%d'), 'cluster': cluster, 'startDatePrediction': startDatePrediIteration, 'NumberWeekPredict': 2,'exog_vars':exog_vars_ ,'rmse': rmse2, 'mape': mape2, 'mae': mae2}
                controls.append(control)
                controls2.append(control2)

c:\Users\herma\anaconda3\envs\modelos\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\herma\anaconda3\envs\modelos\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\herma\anaconda3\envs\modelos\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\herma\anaconda3\envs\modelos\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


2011-02-11 00:00:00
2011-03-11 00:00:00


c:\Users\herma\anaconda3\envs\modelos\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
c:\Users\herma\anaconda3\envs\modelos\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
c:\Users\herma\anaconda3\envs\modelos\lib\site-packages\statsmodels\tsa\statespace\representation.py:374: FutureWarning: Unknown keyword arguments: dict_keys(['S']).Passing unknown keyword arguments will raise a TypeError beginning in version 0.15.
  warnings.warn(msg, FutureWarning)
c:\Users\herma\anaconda3\envs\modelos\lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported

RMSE: 108981.26
MAPE: 0.0315
MAPE (%): 3.15%
MAE: 106409.34
RMSE: 107000.10
MAPE: 0.0308
MAPE (%): 3.08%
MAE: 105460.04


c:\Users\herma\anaconda3\envs\modelos\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\herma\anaconda3\envs\modelos\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\herma\anaconda3\envs\modelos\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\herma\anaconda3\envs\modelos\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\herma\anaconda3\envs\modelos\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finit

...

c:\Users\herma\anaconda3\envs\modelos\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\herma\anaconda3\envs\modelos\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\herma\anaconda3\envs\modelos\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\herma\anaconda3\envs\modelos\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\herma\anaconda3\envs\modelos\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finit

2011-02-18 00:00:00
2011-03-18 00:00:00


c:\Users\herma\anaconda3\envs\modelos\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
c:\Users\herma\anaconda3\envs\modelos\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
c:\Users\herma\anaconda3\envs\modelos\lib\site-packages\statsmodels\tsa\statespace\representation.py:374: FutureWarning: Unknown keyword arguments: dict_keys(['S']).Passing unknown keyword arguments will raise a TypeError beginning in version 0.15.
  warnings.warn(msg, FutureWarning)
c:\Users\herma\anaconda3\envs\modelos\lib\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few o

RMSE: 42077.82
MAPE: 0.0342
MAPE (%): 3.42%
MAE: 35945.47
RMSE: 37767.68
MAPE: 0.0264
MAPE (%): 2.64%
MAE: 27774.61


c:\Users\herma\anaconda3\envs\modelos\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\herma\anaconda3\envs\modelos\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\herma\anaconda3\envs\modelos\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\herma\anaconda3\envs\modelos\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


## ExportFeaturesConfig
This step exports the the feature combination evaluation and is used in the production prediction to know which feature combination is required to use. 

In [ ]:

if flag_cal_best_features == True:
    from openpyxl.styles import Font, Alignment
    dfControl = pd.DataFrame(controls)
    dfControl['run_date'] = pd.to_datetime('now').strftime('%Y-%m-%d')
    dfControl['exog_'] = dfControl['exog_vars'].astype(str)
    dfControl = dfControl.groupby(['startDatePrediction','exog_'])['mae'].sum()
    dfControl = pd.DataFrame(dfControl).reset_index()
    dfControl= dfControl.pivot(columns=['startDatePrediction'], values='mae',index="exog_")
    dfControl = dfControl.reset_index()


    dfControl.columns
    dfControl.index.name = None
    dfControl.columns.name = None
    dfControl= dfControl.set_index('exog_')

    dfControl['mae_sum'] = dfControl.sum(axis=1)
    dfControlMAE = dfControl.sort_values(by='mae_sum')


    dfControlMAE.reset_index(inplace=True)
    dfControlMAE['Choosen'] = np.where(dfControlMAE.index ==0,'True', 'False')
    dfControlMAE['run_date'] = pd.to_datetime('now').strftime('%Y-%m-%d-%H')

    with pd.ExcelWriter("Feature_Selection.xlsx", engine="openpyxl" ) as writer:

        dfControlMAE.rename(columns={'exog_': 'Feature combination for MAE'}, inplace=True)
        dfControlMAE.to_excel(
                writer,
                sheet_name="BEST_FEATURES_COMPARISON",
            
                index=False
            )

## ExportHyperParameterSarima

In [ ]:

if flag_hyper_param_sarima == True:
    df_cluster_hyperparamter= []
    dfWalmartSales = dfWalmartSales_original.copy()
    # NOTE: it's seems that we are duplicating the code from production, but we are doing this for the purpose of understand what uses this part
    ### ReadFeatureSelection
    dfFeatureSelection = pd.read_excel("Feature_Selection.xlsx", sheet_name="BEST_FEATURES_COMPARISON")

    DfPrdFeatures = dfFeatureSelection[dfFeatureSelection['Choosen'] == True]['Feature combination for MAE'].values[0]

    DfPrdFeatures =DfPrdFeatures.replace('[','').replace(']','').replace("'",'').split(',')

    ### ReadClusterAssigment
    DfPrdClusterAssigment = pd.read_excel(os.path.join(output_folder_cluster,"clust_hparam_eval.xlsx"), sheet_name ='Stores_Cluster_Assignment')
    DfPrdClusterAssigment =DfPrdClusterAssigment[['Store', 'cluster_label']]

    dfWalmartSales =dfWalmartSales.merge(DfPrdClusterAssigment, how='left', on='Store')


    for cluster in dfWalmartSales['cluster_label'].unique():
                dfWalmartSalesStore1  = dfWalmartSales[dfWalmartSales['cluster_label'] == cluster]
                dfWalmartSalesStore1 = adjustingData(dfWalmartSalesStore1)

                train, test = trainTest(dfWalmartSalesStore1)
                bestParameterModel = bestHyperparameter(train)   
                df_cluster_hyperparamter.append(

                {'date_run':pd.to_datetime('now').strftime('%Y-%m-%d-%H-%M'), 'cluster':cluster, 'features':json.dumps(DfPrdFeatures), 'hyperparamter':json.dumps(bestParameterModel.get_params())
                }    
                )
                

    pd.DataFrame(df_cluster_hyperparamter).to_excel("Cluster_Hyperparam_sarima.xlsx", index=False)



# Production

## ReadFeatureSelection

In [ ]:
dfFeatureSelection = pd.read_excel("Feature_Selection.xlsx", sheet_name="BEST_FEATURES_COMPARISON")

DfPrdFeatures = dfFeatureSelection[dfFeatureSelection['Choosen'] == True]['Feature combination for MAE'].values[0]

DfPrdFeatures =DfPrdFeatures.replace('[','').replace(']','').replace("'",'').split(',')

DfPrdFeatures =pd.DataFrame(DfPrdFeatures)
DfPrdFeatures.columns = ['features']
DfPrdFeatures['features'] = DfPrdFeatures['features'].str.strip()


## ReadClusterAssigment

In [ ]:
DfPrdClusterAssigment = pd.read_excel(os.path.join(output_folder_cluster,"clust_hparam_eval.xlsx"), sheet_name ='Stores_Cluster_Assignment')
DfPrdClusterAssigment =DfPrdClusterAssigment[['Store', 'cluster_label']]


## ReadHyperParameterSarima

In [ ]:
df_cluster_hyperparam_sarima = pd.read_excel('Cluster_Hyperparam_sarima.xlsx')

## ReadHistoricalPrediction

In [ ]:
df_hist_prediction = pd.read_excel("model_prediction.xlsx", sheet_name ='prediction')
new_id_run = df_hist_prediction['id_run'].max()+1

## NewDataPrediction

In [ ]:
dfPrdWalmartSales = dfWalmartSales_original.copy()
dfPredictionTotalIteration = pd.DataFrame({}, columns=['cluster_no', 'date','predicted_value'])
dfPrdWalmartSales =dfPrdWalmartSales.merge(DfPrdClusterAssigment, how='left', on='Store')
# dfPrdWalmartSales =dfPrdWalmartSales[dfPrdWalmartSales['cluster_label'].isin([1,2])]
for cluster in dfPrdWalmartSales['cluster_label'].unique():
    dfPrdClusterIteration = dfPrdWalmartSales[dfPrdWalmartSales['cluster_label'] == cluster]
    dfPrdClusterIteration = adjustingData(dfPrdClusterIteration)
    train, test = trainTest(dfPrdClusterIteration, startDatePrediction, endDatePrediction)
    dfExog, train = exogenousCalculation(startYearPrediction, StartWeekPredict,NumberWeekPredict, train)
    dfExog =dfExog[:NumberWeekPredict]
    ClusterHyperparams = json.loads(df_cluster_hyperparam_sarima[df_cluster_hyperparam_sarima['cluster'] == cluster].reset_index()['hyperparamter'][0])

    forecast = forecastPrd(ClusterHyperparams['order'], ClusterHyperparams['seasonal_order'],dfExog,exog_vars,NumberWeekPredict,train)
    dfPredictIteration =pd.DataFrame(pd.date_range(startDatePrediction , periods=NumberWeekPredict, freq='W'), columns=['date'])
    dfPredictIteration['cluster_no'] = cluster

    dfPredictIteration = pd.concat([dfPredictIteration.reset_index(drop=True),forecast.reset_index(drop=True)], axis=1)
    dfPredictIteration.rename(columns={'predicted_mean':'predicted_value'}, inplace=True)
    dfPredictionTotalIteration =pd.concat([dfPredictionTotalIteration,dfPredictIteration])


In [ ]:
dfPredictionTotalIteration['date_run'] = pd.to_datetime('now').strftime('%Y-%m-%d')
dfPredictionTotalIteration['id_run'] = new_id_run
dfPredictionTotalIteration =dfPredictionTotalIteration[['id_run', 'date_run','cluster_no', 'date', 'predicted_value']]

df_hist_prediction =pd.concat([df_hist_prediction,dfPredictionTotalIteration])

df_hist_prediction.to_excel("model_prediction.xlsx", sheet_name ='prediction', index=False)

# BackTesting
In this scenario, we assume that new information will be received every week. Therefore, we will perform a backtest using this incoming data.

## NewHistoricalData

In [ ]:
dfBKWalmartSales = dfWalmartSales_original.copy()
# We have added +1 because before it was calculating the same week of prediction as end of the backtesting
startDatePredicBT =startDatePrediction - pd.DateOffset(weeks= BackTestingWeek+1)
endDatePredicBT = startDatePredicBT + pd.DateOffset(weeks= BackTestingWeek)
dfPredictionTotalIteration = pd.DataFrame({}, columns=['cluster_no', 'date','predicted_value'])
dfBKWalmartSales =dfBKWalmartSales.merge(DfPrdClusterAssigment, how='left', on='Store')

dfBacktestingClusters = []

for cluster in dfBKWalmartSales['cluster_label'].unique():
    dfPrdClusterIteration = dfBKWalmartSales[dfBKWalmartSales['cluster_label'] == cluster]
    dfPrdClusterIteration = adjustingData(dfPrdClusterIteration)
    train, test = trainTest(dfPrdClusterIteration,startDatePredicBT, endDatePredicBT)
   

    dfExog, train = exogenousCalculation(startYearPrediction, StartWeekPredict,NumberWeekPredict, train)
    dfExog =dfExog[:NumberWeekPredict]
    ClusterHyperparams = json.loads(df_cluster_hyperparam_sarima[df_cluster_hyperparam_sarima['cluster'] == cluster].reset_index()['hyperparamter'][0])

    forecast = forecastPrd(ClusterHyperparams['order'], ClusterHyperparams['seasonal_order'],dfExog,exog_vars,BackTestingWeek,train)
    rmse, mape, mae,predicted_value, expected_value = metricts(test,NumberWeekPredict, forecast)

    dfBacktestingClusters.append({ 'start_date':startDatePredicBT, 'end_date':endDatePredicBT, 'date_run':pd.to_datetime('now').strftime('%Y-%m-%d') , 
                                  'cluster':cluster , 'rmse': rmse, 'mape': mape, 'mae': mae, 'expected_value':expected_value, 'predicted_value':predicted_value, 'bias%': round((expected_value-predicted_value)/expected_value,4) })

## ReadBackTestingHistorical

In [ ]:

clusterSummaryHist = pd.read_excel(os.path.join(os.getcwd(), 'backtesting_logs.xlsx' ), sheet_name='cluster_summary')
summaryHist = pd.read_excel(os.path.join(os.getcwd(), 'backtesting_logs.xlsx' ), sheet_name='summary')

# Ensured that the date has the right format
clusterSummaryHist['end_date'] = pd.to_datetime(clusterSummaryHist['end_date'])
clusterSummaryHist['date_run'] = pd.to_datetime(clusterSummaryHist['date_run'])
summaryHist['end_date'] = pd.to_datetime(summaryHist['end_date'])
summaryHist['date_run'] = pd.to_datetime(summaryHist['date_run'])


# We are excluding if there are already logs to avoid duplicating the information
clusterSummaryHist = clusterSummaryHist[~(clusterSummaryHist['end_date'] ==endDatePredicBT)]
summaryHist = summaryHist[~(summaryHist['end_date'] ==endDatePredicBT)]

## ExportHistoricalBackTesting

In [ ]:
dfBacktestingClusters =pd.DataFrame(dfBacktestingClusters)
BackTestingSummary = pd.DataFrame(dfBacktestingClusters.groupby(['start_date','end_date', 'date_run']).agg({'mae':'mean', 'mape':'mean', 'expected_value':'sum', 'predicted_value':'sum'})  ).reset_index()

BackTestingSummary['bias%'] = (BackTestingSummary['expected_value']  - BackTestingSummary['predicted_value'] ) / BackTestingSummary['expected_value']
BackTestingSummary['threshold_bias%'] = 0.10
BackTestingSummary['status_threshold%']  = np.where ( BackTestingSummary['threshold_bias%']>BackTestingSummary['bias%'] , 'OK', 'ERROR' )
clusterSummaryHist =pd.concat([clusterSummaryHist,dfBacktestingClusters])
summaryHist =pd.concat([summaryHist,BackTestingSummary])

with pd.ExcelWriter('backtesting_logs.xlsx' ,engine='openpyxl') as writer: 
    summaryHist.to_excel(writer, sheet_name = 'summary', index=False)
    clusterSummaryHist.to_excel(writer, sheet_name='cluster_summary', index=False)
    